# 第7回 演習: テキスト処理と文章生成

課題1〜3 で、英文を読んで「ある語の次に来やすい語」を数え、それを使って文章をでたらめに作る機械 (マルコフ連鎖) を作る。
課題4 では、同じ「表を引いて次の状態を決める」形で、電卓を作る。

## このノートブックの使い方

- **上のセルから順に実行する** (`Shift + Enter`)。途中から実行すると、前のセルで作った名前が無いので `NameError` (名前が見つからない) になる
- 自分で書くのは、`TODO` と書かれた所だけ。そこにある `pass` や `...` を消して、コードに置き換える
- 各課題の下にある `# [CHECK]` のセルは**確認用。書き換えない**。実行して `OK` と出れば正解
- `NG` や `AssertionError` が出たら、その行の後ろのメッセージ (例: `'小文字にそろえること'`) が直すべき点
- 各課題は「説明を読む → 例題セルを実行 → `TODO` を埋める → 確認セルを実行」の順に進める

## 全体の地図

前の課題で作った関数を、次の課題で使う (`tokenize_file` → `build_table` → `generate`)。

| 課題 | 書くもの | 使う道具 | 配点 |
|---|---|---|---|
| 課題1 | `tokenize_file` | `with open`, `re.findall` | 20 |
| 課題2 | `build_table` | `zip`, `defaultdict(Counter)` | 15 |
| 課題2-b | `top_following` | `sorted(key=...)` | 10 |
| 課題3 | `pick_next`, `generate` | `random.choices`, ループ | 20 |
| 課題4 | `evaluate`, `- * /` | 関数を値に持つ辞書 | 15 |
| 課題4-b | スタック操作の関数 | `WORDS['DUP'] = dup` で表に足す | 10 |
| 発展1・2 | 温度、2語を見る表 | 余力があれば | 各5 |

## 提出まで

- 最初に、下の2つの準備セルを実行する
- 終わったら、出力を残したまま保存し、kibaco から提出する

In [ ]:
# 準備 (最初に実行)
import urllib.request, pathlib

SAMPLE = '''I like cats. I like dogs. I love cats.
The cat sat on the mat. The dog sat on the log.

# コメント行 (課題1 で無視する)
A quick brown fox jumps over the lazy dog!
The quick brown dog meets a lazy fox.
'''
with open('sample.txt', 'w', encoding='utf-8') as f:
    f.write(SAMPLE)

# 長めの英文 (取れなくても先へ進める)
BOOK = 'alice.txt'
if pathlib.Path(BOOK).exists():
    print('長文テキストは手元にあります:', BOOK)
else:
    try:
        urllib.request.urlretrieve(
            'https://www.gutenberg.org/files/11/11-0.txt', BOOK)
        print('長文テキストを取得しました:', BOOK)
    except Exception as e:
        with open(BOOK, 'w', encoding='utf-8') as f:
            f.write(SAMPLE * 50)
        print('取得できなかったのでサンプルで代用します:', e)

print('準備完了')


In [ ]:
#@title 自動セットアップ (実行するだけ。中身は隠しています)
# [SKIP] 採点の対象外
# ---- 自己確認の準備 (check / score)。1回実行すればよい ----
import pathlib
import urllib.request

if not pathlib.Path('autograde.py').exists():
    try:
        urllib.request.urlretrieve(
            'https://3tty0n.github.io/2026-scripting-lab/grading/autograde.py', 'autograde.py')
    except Exception as e:
        print('取得できませんでした:', e)

from autograde import check, score
TASK = '07'


---
## 課題1: 文章を単語に分ける (20点)

**やること**: ファイルを1行ずつ読み、**小文字の単語のリスト**を返す関数 `tokenize_file(path)` を書く。
文章を単語の並びに分けることを、トークン化と呼ぶ。

**入出力の例** (`sample.txt` は準備セルで作ったファイル):

```
tokenize_file('sample.txt')  →  ['i', 'like', 'cats', 'i', 'like', 'dogs', ...]  (全部で 38 語)
```

**手順** (コードの `TODO 1`・`TODO 2` に対応):

1. その行が `#` で始まっていたら、`continue` で次の行へ進む (コメント行は使わない)
2. 行を小文字にし、`re.findall(r"[a-z']+", ...)` で単語だけを取り出して、`tokens.extend(...)` で `tokens` の後ろにつなげる

**ヒント**: `re` は正規表現 (文字の並びのパターン) を扱う道具。
`r"[a-z']+"` は「英小文字か `'` が1文字以上続く所」という意味で、`.` `,` `!` などの記号は取り出されない。
`line.split()` (空白で区切るだけ) だと、`'dog!'` のように記号がくっついたまま残る。

**つまずきやすい点**:
- `'小文字にそろえること'` と出る → `line.lower()` を忘れている
- `'… 語。区切り方を見直すこと'` と出る → `split()` を使っている、または `#` の行を飛ばしていない
- `append` を使うと、リストの中にリストが入ってしまう。`extend` を使う

### 例題

次のセルを実行して、動きを見る。

まず、部品を1つずつ確かめる。


In [ ]:
# 例題
import re

line = "The dog's bone!  A cat."
print(line.startswith('#'))    # False
print(re.findall(r"[a-z']+", line.lower()))
# ["the", "dog's", "bone", 'a', 'cat']

ファイルを1行ずつ読む形はこれ。`with open(...) as f` で開くと、使い終わったら自動で閉じる。


In [ ]:
# 例題
found = []
with open('sample.txt', encoding='utf-8') as f:
    for line in f:
        if line.startswith('#'):
            continue
        found.extend(re.findall(r"[a-z']+", line.lower()))
print(found[:5])

`extend` はリストの中身を**つなげる**。`append` だと、リストの中にリストが1つの要素として入ってしまう。


In [ ]:
import re


def tokenize_file(path):
    tokens = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            # TODO 1: line が '#' で始まるなら continue
            # TODO 2: line.lower() から re.findall で単語を取り出し、tokens.extend でつなげる
            pass
    return tokens


toks = tokenize_file('sample.txt')
print(len(toks), toks[:12])

In [ ]:
# [CHECK] q1
need('tokenize_file', arity=1)

toks = tokenize_file('sample.txt')
assert isinstance(toks, list)
assert toks[:6] == ['i', 'like', 'cats', 'i', 'like', 'dogs'], toks[:6]
assert 'これはコメント行' not in ''.join(toks), 'コメント行を無視すること'
assert 'dog!' not in toks and 'dog' in toks, '記号を落とすこと'
assert 'cats.' not in toks
assert all(t == t.lower() for t in toks), '小文字にそろえること'
assert len(toks) == 38, f'{len(toks)} 語。区切り方を見直すこと'
assert toks[-1] == 'fox', toks[-1]
print('OK')


---
## 課題2: 「次に来た語」の表 (15点)

**やること**: 単語のリストから、「ある語の次に、どの語が何回来たか」の表を作る関数 `build_table(tokens)` を書く。

**入出力の例**:

```
build_table(['i', 'like', 'cats', 'i', 'like', 'dogs'])
  →  'i'    の次は {'like': 2}
     'like' の次は {'cats': 1, 'dogs': 1}
     'cats' の次は {'i': 1}
```

**手順** (コードの `TODO` に対応):

1. `for a, b in zip(tokens, tokens[1:]):` で、隣り合う2語 (前の語 `a`、後の語 `b`) を順に取り出し、`table[a][b] += 1` で1増やす

**ヒント**: `zip(tokens, tokens[1:])` は、リストと「1つずらしたリスト」を並べて、隣り合う2語の組を作る。
`defaultdict(Counter)` は「まだ無い語を引くと、空の数え表 (`Counter`) を自動で用意してくれる辞書」。
だから、初めて出てきた語でも、準備なしで `table[a][b] += 1` と書ける。

**つまずきやすい点**:
- `'1語だけならペアは作れない'` と出る → `zip` を使わず、自分で添字を数えてはみ出している
- `table` を `return` する行は、もう書いてある。消さないこと

### 例題

次のセルを実行して、動きを見る。

隣り合う2語の組を作って、数える。


In [ ]:
# 例題
from collections import defaultdict, Counter

toks = ['i', 'like', 'cats', 'i', 'like', 'dogs']
print(list(zip(toks, toks[1:])))
# [('i','like'), ('like','cats'), ('cats','i'), ('i','like'), ('like','dogs')]

table = defaultdict(Counter)
table['i']['like'] += 1
table['i']['like'] += 1
print(dict(table['i']))    # {'like': 2}

あとは、組を1つずつ取り出して `table[前][後] += 1` するだけ。


In [ ]:
from collections import defaultdict, Counter


def build_table(tokens):
    table = defaultdict(Counter)
    # TODO: zip(tokens, tokens[1:]) で隣り合う2語 a, b を取り出し、table[a][b] += 1
    return table


table = build_table(tokenize_file('sample.txt'))
for w in ['i', 'the', 'like']:
    print(f'{w:6} -> {dict(table[w])}')

In [ ]:
# [CHECK] q2
need('build_table', arity=1)

table = build_table(tokenize_file('sample.txt'))
assert dict(table['i']) == {'like': 2, 'love': 1}, dict(table['i'])
assert dict(table['like']) == {'cats': 1, 'dogs': 1}
assert table['the']['quick'] == 1
assert sum(table['the'].values()) == 6, 'the の次に来た語は全部で6回'
assert len(build_table([])) == 0
assert len(build_table(['a'])) == 0, '1語だけならペアは作れない'
assert dict(build_table(['a', 'b', 'a', 'b'])['a']) == {'b': 2}

# 語をいくつ並べても、表に入る回数の合計は (語数 - 1) になるはず
holds('ペアの総数は語数-1',
      lambda ws: sum(sum(c.values()) for c in build_table(ws).values())
                 == max(len(ws) - 1, 0),
      [words(n) for n in range(0, 12)])
print('OK')


### 課題2-b: 続きやすい語 (10点)

**やること**: 課題2 の表から、「ある語の次に来やすい語」を多い順に `n` 件返す関数 `top_following(table, word, n)` を書く。

**入出力の例** (`t` は `sample.txt` から作った表):

| 呼び方 | 返り値 |
|---|---|
| `top_following(t, 'i', 5)` | `[('like', 2), ('love', 1)]` |
| `top_following(t, 'like', 5)` | `[('cats', 1), ('dogs', 1)]` (同じ回数なら ABC 順) |
| `top_following(t, 'zzz', 5)` | `[]` (表に無い語) |

**手順** (コードの `TODO 1`・`TODO 2` に対応):

1. `word` が `table` に無ければ、空のリスト `[]` を返す
2. `table[word].items()` を、回数の多い順 (同じ回数なら単語の ABC 順) に `sorted` で並べ、`[:n]` で先頭 `n` 件を返す

**ヒント**: `sorted(..., key=lambda kv: (-kv[1], kv[0]))` と書く。
`kv` は `('like', 2)` のような (単語, 回数) の組。`-kv[1]` で回数の大きい順、`kv[0]` で同じ回数なら ABC 順になる。
並び順を決めておかないと、同じ回数の語の順番が実行のたびに変わってしまう。

**つまずきやすい点**:
- `'同数なら辞書順'` と出る → `key` に `kv[0]` を入れ忘れている
- `'無い語なら空リスト'` と出る → 手順1 を書いていない。何も返さないと `None` になる

### 例題

次のセルを実行して、動きを見る。

回数の多い順、同じ回数なら ABC 順に並べる。


In [ ]:
# 例題
from collections import Counter

c = Counter({'cat': 3, 'zoo': 1, 'ant': 1})
print(sorted(c.items(), key=lambda kv: (-kv[1], kv[0])))
# [('cat', 3), ('ant', 1), ('zoo', 1)]


先頭から `n` 件を切り出すには `[:n]` と書く。表に無い語かどうかは `word not in table` で確かめる。


In [ ]:
def top_following(table, word, n=5):
    # TODO 1: word が table に無ければ [] を返す
    # TODO 2: table[word].items() を (-回数, 単語) の順で sorted し、[:n] を返す
    pass


big_tokens = tokenize_file(BOOK)
big_table = build_table(big_tokens)
print(f'{len(big_tokens):,} 語 / 異なり語数 {len(set(big_tokens)):,}')
print('the の次:', top_following(big_table, 'the', 5))

In [ ]:
# [CHECK] q2b
need('top_following', arity=2)

t = build_table(tokenize_file('sample.txt'))
assert top_following(t, 'i', 5) == [('like', 2), ('love', 1)]
assert top_following(t, 'like', 5) == [('cats', 1), ('dogs', 1)], '同数なら辞書順'
assert top_following(t, 'i', 1) == [('like', 2)]
assert top_following(t, 'zzz', 5) == [], '無い語なら空リスト'

r = top_following(big_table, 'the', 5)
assert len(r) == 5 and all(isinstance(w, str) and isinstance(c, int) for w, c in r)
assert [c for _w, c in r] == sorted([c for _w, c in r], reverse=True), '多い順に'
print('OK')


---
## 課題3: 表を使って文章を作る (20点)

**やること**: 2つの関数を書く。

- `pick_next(counter)`: 回数を「選ばれやすさ」(重み) にして、次の語を1つ選ぶ
- `generate(table, start, n)`: 語 `start` から始めて、次の語を選んでは後ろにつなげる。**最大 `n` 語**つなげ、空白で区切った1つの文字列を返す。次の語が表に無い語に来たら、そこで終わる

このように「今の語だけを見て、次の語をくじで選ぶ」ことを繰り返す仕組みを、マルコフ連鎖と呼ぶ。

**入出力の例**:

```
pick_next(Counter({'like': 2, 'love': 1}))  →  'like' (2/3 の確率) か 'love' (1/3 の確率)
generate(table, 'i', 10)                     →  'i like cats i love cats ...' (毎回変わる)
```

**手順** (コードの `TODO` に対応):

- `pick_next`: `random.choices(語のリスト, weights=回数のリスト)[0]` を返す
- `generate` のループの中:
  1. 今の語 `word` が `table` に無い (または次の語が1つも無い) なら、`break` でループを抜ける
  2. `word = pick_next(table[word])` で次の語に進む
  3. `result.append(word)` で、選んだ語を `result` に足す

**ヒント**: `random.choices(...)` は**リストを返す**ので、最後に `[0]` を付けて1語を取り出す。
`random.choice` (s が無い方) は、どの語も同じ確率で選ぶので**使わないこと**。

**つまずきやすい点**:
- `'回数を重みにすること'` と出る → `weights=` を渡していないか、`random.choice` を使っている
- `'開始語 + 最大 n 語'` と出る → ループを `n` 回より多く回している
- `generate(dead, 'b', 10)` で止まる、エラーになる → 手順1 の `break` が無い

### 例題

次のセルを実行して、動きを見る。

重みつきのくじ引き。


In [ ]:
# 例題
import random
from collections import Counter

c = Counter({'like': 3, 'love': 1})
print(random.choices(list(c.keys()), weights=list(c.values()))[0])
# 3/4 の確率で 'like'


つなげる側の形。小さな表 `demo_table` で確かめる (本物の表は `build_table` で作ったもの)。
`range(n)` で最大 `n` 回まわし、次の語が無ければ `break` で抜ける。


In [ ]:
# 例題
demo_table = {'the': {'cat': 2, 'dog': 1}, 'cat': {'sat': 1}, 'dog': {'sat': 1}}
n = 5

word = 'the'
result = [word]
for _ in range(n):
    if word not in demo_table or not demo_table[word]:
        break
    c = demo_table[word]
    word = random.choices(list(c), weights=list(c.values()))[0]
    result.append(word)
print(' '.join(result))    # 例: the cat sat

In [ ]:
import random


def pick_next(counter):
    # TODO: random.choices(語のリスト, weights=回数のリスト)[0] を返す
    pass


def generate(table, start, n=30):
    word = start
    result = [word]
    for _ in range(n):
        # TODO 1: word が table に無い (または次の語が無い) なら break
        # TODO 2: word = pick_next(table[word]) で次の語へ進む
        # TODO 3: result.append(word)
        pass
    return ' '.join(result)


random.seed(0)
for _ in range(3):
    print(generate(big_table, 'the', 25))
    print()

In [ ]:
# [CHECK] q3
need('pick_next', arity=1)
need('generate', arity=2)

from collections import Counter as _C
c = _C({'like': 2, 'love': 1})
assert set(pick_next(c) for _ in range(200)) == {'like', 'love'}, '両方選ばれること'
picks = [pick_next(c) for _ in range(600)]
assert 300 < picks.count('like') < 500, f"回数を重みにすること (like が {picks.count('like')}/600)"
assert pick_next(_C({'only': 5})) == 'only'

small = build_table(tokenize_file('sample.txt'))
out = generate(small, 'i', 10)
assert isinstance(out, str) and out.startswith('i '), out
assert len(out.split()) <= 11, '開始語 + 最大 n 語'
assert generate(small, 'i', 0) == 'i'

# 次が無い語で止まること
dead = build_table(['a', 'b'])
assert generate(dead, 'b', 10) == 'b'
assert generate(dead, 'a', 10) == 'a b'

# 表に沿った語しか出ないこと
chain = build_table(['x', 'y', 'x', 'y', 'x'])
for _ in range(20):
    ws = generate(chain, 'x', 6).split()
    assert all(a != b for a, b in zip(ws, ws[1:])), 'x と y が交互になるはず'
# 生成しても表そのものは壊さないこと
unchanged('generate', small, 'i', 5)
note(f'sample.txt の語 {len(small)} 種類から文を作れている')

print('OK')


---
## 課題4: 表を引く電卓 (15点)

**やること**: `3 4 +` のように「数を先に、記号を後に書く」式 (逆ポーランド記法) を計算する電卓を作る。

- 数は、スタック (積み上げた数の山。Python ではただのリスト) に積む。`append` で一番上に積み、`pop` で一番上から取り出す
- 記号は、表 `WORDS` で「記号 → スタックを操作する関数」を引いて、その関数を呼ぶ
- 文章生成では「語 → 次の語」の表を引いた。ここでは「記号 → 関数」の表を引く。表に関数を入れる形は、第6回の `COMMANDS` と同じ

**入出力の例**:

| 呼び方 | 返り値 | 意味 |
|---|---|---|
| `calc_rpn('3 4 +')` | `[7]` | 3 + 4 |
| `calc_rpn('10 3 -')` | `[7]` | 10 − 3 |
| `calc_rpn('3 4 + 2 *')` | `[14]` | (3 + 4) × 2 |
| `calc_rpn('20 4 /')` | `[5]` | 20 ÷ 4 (整数の割り算) |

**手順** (コードの `TODO 1`・`TODO 2` に対応):

1. `+` の `add` を真似して、`-` (`sub`)、`*` (`mul`)、`/` (`div`) の関数を書き、`WORDS['-'] = sub` のように表に入れる。割り算は整数の割り算 `//` を使う
2. `evaluate` のループの中で、`token` (区切った1つ分の文字列) が
   - `WORDS` にあれば: `WORDS[token](stack)` で関数を呼ぶ
   - 数なら (`token.lstrip('-').isdigit()` が真なら): `stack.append(int(token))` で積む
   - どちらでもなければ: `raise ValueError(...)` でエラーにする

**ヒント**: 表に入れる関数は、**スタック1つだけを受け取る**。
引き算と割り算は**順番に注意**。`b, a = s.pop(), s.pop()` と書くと、先に取り出した `b` が右側の数になる。`10 3 -` は `10 - 3`。
`token.lstrip('-')` は先頭の `-` を外す。これで `-5` のような負の数も数として扱える。

**つまずきやすい点**:
- `10 3 -` が `[-7]` になる → `a - b` と `b - a` が逆
- `'知らないワードは ValueError を出すこと'` と出る → 手順2 の3つ目が無い
- `'実行中にワードを増やせること'` と出る → `evaluate` の中で `if token == '+'` のように記号を直接書いている。必ず `WORDS` を引く

### 例題

次のセルを実行して、動きを見る。

関数を書いて、表に入れる形。ここでは練習用の表 `DEMO` に入れる (課題4 では `WORDS` に入れる)。


In [ ]:
# 例題
DEMO = {}


def neg(s):
    s.append(-s.pop())


DEMO['NEG'] = neg

s = [3]
DEMO['NEG'](s)
print(s)    # [-3]


`evaluate` の形。練習用の表 `DEMO` を引く版。課題4 では `DEMO` を `WORDS` に変える。


In [ ]:
# 例題
def evaluate_demo(stack, text):
    for token in text.split():
        if token in DEMO:
            DEMO[token](stack)
        elif token.lstrip('-').isdigit():
            stack.append(int(token))
        else:
            raise ValueError(f'知らないワード: {token}')


s = []
evaluate_demo(s, '3 NEG 5')
print(s)    # [-3, 5]

In [ ]:
WORDS = {}


def add(s):
    b, a = s.pop(), s.pop()
    s.append(a + b)


WORDS['+'] = add

# TODO 1: add を真似して sub (-), mul (*), div (/ は //) を書き、WORDS['-'] = sub のように入れる


def evaluate(stack, text):
    for token in text.split():
        # TODO 2: WORDS にあれば WORDS[token](stack)、数なら stack.append(int(token))、それ以外は ValueError
        pass


def calc_rpn(text):
    s = []
    evaluate(s, text)
    return s


print(calc_rpn('3 4 + 2 *'))

In [ ]:
# [CHECK] q4
need('WORDS', kind=dict)
need('evaluate', arity=2)

assert calc_rpn('3 4 +') == [7]
assert calc_rpn('10 3 -') == [7]        # 順序に注意
assert calc_rpn('3 4 + 2 *') == [14]
assert calc_rpn('2 3 4 * +') == [14]
assert calc_rpn('20 4 /') == [5]
assert calc_rpn('-5 3 +') == [-2]
assert calc_rpn('1 2 3') == [1, 2, 3]
assert calc_rpn('') == []

# 性質で確かめる: 数を積むだけなら、その順に並ぶ
holds('数を並べたらそのまま積まれる',
      lambda a, b: calc_rpn(f'{a} {b}') == [a, b], ints(2, lo=0, hi=99))
holds('足してから引くと元の数',
      lambda a, b: calc_rpn(f'{a} {b} + {b} -') == [a], ints(2, lo=0, hi=99))

# WORDS に後から足したワードも、evaluate を変えずに使えること
def _neg(s):
    s.append(-s.pop())
WORDS['NEG'] = _neg
assert calc_rpn('5 NEG') == [-5], '実行中にワードを増やせること'
del WORDS['NEG']

try:
    calc_rpn('3 FOO')
except ValueError:
    pass
else:
    raise AssertionError('知らないワードは ValueError を出すこと')
print('OK')

---
### 課題4-b: スタックを組み替える関数 (10点)

**やること**: スタックの数の並びを組み替える5つの関数を書き、`WORDS` に入れる (表に入れる行はもう書いてある)。
**`evaluate` は書き換えない**。表に足すだけで電卓が使える記号が増えるのが、表を引く形のよいところ。

**入出力の例**: 下の表の「スタックの変化」は、左が実行前、右が実行後 (右端が一番上)。

| 記号 | 関数 | 動作 | スタックの変化 | 例 |
|---|---|---|---|---|
| `DUP` | `dup` | 一番上を複製 | (a → a a) | `5 DUP` → `[5, 5]` |
| `DROP` | `drop` | 一番上を捨てる | (a → ) | `1 2 DROP` → `[1]` |
| `SWAP` | `swap` | 上の2つを入れ替える | (a b → b a) | `1 2 SWAP` → `[2, 1]` |
| `OVER` | `over` | 上から2番目をコピーして積む | (a b → a b a) | `1 2 OVER` → `[1, 2, 1]` |
| `ROT` | `rot` | 上から3番目を一番上へ移す | (a b c → b c a) | `1 2 3 ROT` → `[2, 3, 1]` |

**手順**: 各関数の `TODO` を、1〜3行で書く。

**ヒント**: 先に紙に図を書いてから、コードにする。
`s[-1]` はリストの一番後ろ (スタックの一番上) を、取り出さずに見る。`s[-2]` は上から2番目。
`SWAP` は、2つ取り出してから、逆の順番に積む。

**つまずきやすい点**:
- `'課題4 が壊れていないこと'` と出る → 課題4 のセルを実行し直していない。課題4 から順に実行し直す
- `ROT` は3つ取り出すと分かりやすい: `c, b, a = s.pop(), s.pop(), s.pop()` (`c` が一番上)


In [ ]:
def dup(s):
    # TODO: 一番上 s[-1] をもう1つ積む
    pass


WORDS['DUP'] = dup


def drop(s):
    # TODO: 一番上を pop して捨てる
    pass


WORDS['DROP'] = drop


def swap(s):
    # TODO: 2つ pop して、逆の順番に積む
    pass


WORDS['SWAP'] = swap


def over(s):
    # TODO: 上から2番目 s[-2] を積む
    pass


WORDS['OVER'] = over


def rot(s):
    # TODO: 3つ pop して、a b c の順が b c a になるように積む
    pass


WORDS['ROT'] = rot


print(calc_rpn('5 DUP *'))      # [25]
print(calc_rpn('1 2 3 ROT'))    # [2, 3, 1]


In [ ]:
# [CHECK] q4b
need('WORDS', kind=dict)
need('evaluate', arity=2)

assert calc_rpn('5 DUP *') == [25]
assert calc_rpn('1 2 DROP') == [1]
assert calc_rpn('1 2 SWAP') == [2, 1]
assert calc_rpn('1 2 OVER') == [1, 2, 1]
assert calc_rpn('1 2 3 ROT') == [2, 3, 1]
assert calc_rpn('1 2 3 ROT DROP SWAP') == [3, 2]
assert calc_rpn('3 4 + DUP *') == [49], '課題4 が壊れていないこと'

holds('SWAP を2回かけると元に戻る',
      lambda a, b: calc_rpn(f'{a} {b} SWAP SWAP') == [a, b], ints(2, lo=0, hi=99))
holds('DUP して DROP すると元に戻る',
      lambda a: calc_rpn(f'{a} DUP DROP') == [a], ints(1, lo=0, hi=99))
print('OK')

**次回 (第8回) に向けて**: 課題4・4-b で書いた `WORDS` / `evaluate` と各関数は、
第8回 の Forth インタプリタの土台として**そのまま貼って使う** (名前を変えないこと)。
手元にコピーを残しておくこと。


---
# 発展課題

余力があれば。


## 発展1: 温度 (5点)

**やること**: 選び方の「かたより」を、温度という数で調節できるようにする。

- `pick_next_t(counter, temperature)`: 重みを `回数 ** (1 / temperature)` に変えてから、`pick_next` と同じように1語選ぶ
- `generate_t(table, start, n, temperature)`: 課題3 の `generate` と同じ。ただし次の語は `pick_next_t` で選ぶ

**ヒント**: 温度が小さい (例: 0.2) と回数の差が広がり、いちばん多い語ばかり選ばれる。
温度が大きい (例: 3.0) と差が縮み、どの語もほぼ同じ確率になる。

### 例題

次のセルを実行して、動きを見る。

重みを作り替えてからくじを引くだけ。


In [ ]:
# 例題
counts = [3, 1]
t = 0.5
print([c ** (1 / t) for c in counts])    # [9.0, 1.0]
t = 3.0
print([c ** (1 / t) for c in counts])    # [1.44, 1.0]

`generate_t` は課題3 の `generate` と同じ形。次の語を選ぶ所だけ `pick_next_t` に変える。


In [ ]:
def pick_next_t(counter, temperature=1.0):
    # TODO: 重みを 回数 ** (1 / temperature) にして random.choices で1語選ぶ
    pass


def generate_t(table, start, n=30, temperature=1.0):
    # TODO: generate と同じ。次の語は pick_next_t で選ぶ
    pass


random.seed(0)
for t in [0.2, 1.0, 3.0]:
    print(f'--- temperature = {t} ---')
    print(generate_t(big_table, 'the', 25, temperature=t))
    print()


In [ ]:
# [CHECK] b1
need('pick_next_t', arity=1)
need('generate_t', arity=2)

from collections import Counter as _C
c = _C({'a': 9, 'b': 1})
cold = [pick_next_t(c, 0.2) for _ in range(200)]
hot  = [pick_next_t(c, 5.0) for _ in range(200)]
assert cold.count('a') > 195, f'低温では最頻語に偏るはず (a が {cold.count("a")}/200)'
assert 60 < hot.count('a') < 175, f'高温では平坦に近づくはず (a が {hot.count("a")}/200)'
assert pick_next_t(_C({'only': 3}), 0.5) == 'only'

out = generate_t(big_table, 'the', 5, 1.0)
assert isinstance(out, str) and out.startswith('the'), out
assert len(out.split()) <= 6
print('OK')


## 発展2: 直前の2語を見る表 (5点)

**やること**: 直前の1語ではなく**2語**を見て次の語を選ぶと、より自然な文になる。そのための2つの関数を書く。

- `build_table2(tokens)`: 課題2 と同じ表。ただし鍵 (表を引く見出し) を2語の組 `(a, b)` にする
- `generate2(table, start_pair, n)`: 2語 `start_pair` から始めて、最大 `n` 語つなげる

**入出力の例**:

```
build_table2(['a', 'b', 'c', 'a', 'b', 'd'])  →  ('a', 'b') の次は {'c': 1, 'd': 1}
```

**ヒント**: `zip(tokens, tokens[1:], tokens[2:])` で、隣り合う3語の組が取れる。

### 例題

次のセルを実行して、動きを見る。

3語の組の取り方。


In [ ]:
# 例題
toks = ['a', 'b', 'c', 'd']
print(list(zip(toks, toks[1:], toks[2:])))
# [('a','b','c'), ('b','c','d')]


表に足すときは `table[(a, b)][c] += 1` と、鍵を2語の組にする。
文を作る側は直前の2語を `pair` に覚えておき、1語進むたびに `pair = (pair[1], 選んだ語)` と更新する。


In [ ]:
def build_table2(tokens):
    # TODO: 鍵を (a, b) にした表を作る。zip(tokens, tokens[1:], tokens[2:]) を使う
    pass


def generate2(table, start_pair, n=30):
    # TODO: generate と同じ。直前2語 pair で表を引き、pair = (pair[1], 選んだ語) と進める
    pass


big_table2 = build_table2(big_tokens)
random.seed(0)
print('bigram :', generate(big_table, 'the', 25))
print()
print('trigram:', generate2(big_table2, ('the', 'queen'), 25))


In [ ]:
# [CHECK] b2
need('build_table2', arity=1)
need('generate2', arity=2)

t2 = build_table2(['a', 'b', 'c', 'a', 'b', 'd'])
assert dict(t2[('a', 'b')]) == {'c': 1, 'd': 1}, dict(t2[('a', 'b')])
assert dict(t2[('b', 'c')]) == {'a': 1}
assert len(build_table2(['a', 'b'])) == 0, '2語では3つ組は作れない'

out = generate2(t2, ('a', 'b'), 5)
assert out.startswith('a b '), out
assert len(out.split()) <= 7, '開始2語 + 最大 n 語'

# trigram のほうが組み合わせは多い
assert len(big_table2) > len(big_table), '組み合わせの数は増えるはず'

# 元の文に現れた並びしか出ないこと
src = ['p', 'q', 'r', 's', 'p', 'q', 'r', 's']
t3 = build_table2(src)
for _ in range(10):
    ws = generate2(t3, ('p', 'q'), 6).split()
    for a, b, c in zip(ws, ws[1:], ws[2:]):
        assert c in t3[(a, b)], f'{(a, b, c)} は元の文に無い並び'
print('OK')


---
## 自己確認

`score()` を実行すると、全部の課題をまとめて確かめられる。何度実行してもよい。
`NG` の行に、次に何を直せばよいかが出る。


In [ ]:
# [SKIP] 採点の対象外
score()


---
## 提出

提出は kibaco から。`score()` は、提出前の確認用。

1. `ランタイム → すべてのセルを実行` で、上から全部実行し、出力を残す
2. `ファイル → .ipynb をダウンロード` で保存する
3. 締切までに kibaco へ提出する

確認セルが全部 `OK` なら提出してよい。確認セルを消しても、教員の採点は別の検査で行う。